# PyraClaw Freedom Engine — GPU/TPU/NPU Launch Pad
## DD7 International GmbH | Patent: PCT/EP2025/080977
## ORCID: 0009-0001-9561-5483 | Byron Callaghan

**MINTED_GREEN | SURGICAL | Sovereign Guard Active**

This notebook connects to:
- Google Drive 2TB (data lake)
- Vertex AI (NPU/TPU inference)
- NemoClaw x3 (GPU inference via NVIDIA)
- PyraClaw services (QDP, RSFS, Evidence)

---

In [ ]:
# ── Step 1: Mount Google Drive (2TB) ──────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Set workspace path
import os
DRIVE_ROOT = '/content/drive/MyDrive'
PYRACLAW_DIR = f'{DRIVE_ROOT}/PyraClaw'
EVIDENCE_DIR = f'{PYRACLAW_DIR}/evidence'
MODELS_DIR = f'{PYRACLAW_DIR}/models'
DATA_DIR = f'{PYRACLAW_DIR}/data'

for d in [PYRACLAW_DIR, EVIDENCE_DIR, MODELS_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Drive mounted. PyraClaw workspace: {PYRACLAW_DIR}')
!df -h /content/drive | tail -1

In [ ]:
# ── Step 2: Check GPU/TPU ─────────────────────────────────────────
import subprocess

# GPU check
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                            '--format=csv,noheader'], capture_output=True, text=True)
    if result.returncode == 0:
        print(f'GPU: {result.stdout.strip()}')
    else:
        print('No NVIDIA GPU detected — CPU mode')
except FileNotFoundError:
    print('nvidia-smi not found — CPU mode')

# TPU check
try:
    import jax
    devices = jax.devices()
    print(f'JAX devices: {[str(d) for d in devices]}')
except ImportError:
    print('JAX not installed — no TPU')

# Memory
import psutil
mem = psutil.virtual_memory()
print(f'RAM: {mem.total / (1024**3):.1f} GB ({mem.percent}% used)')

In [ ]:
# ── Step 3: Install PyraClaw dependencies ────────────────────────
!pip install -q fastapi uvicorn httpx pydantic numpy aiohttp

# Clone PyraClaw repo
!git clone https://github.com/tritathadore/pyraclaw.git /content/pyraclaw 2>/dev/null || \
    (cd /content/pyraclaw && git pull)

import sys
sys.path.insert(0, '/content/pyraclaw')
sys.path.insert(0, '/content/pyraclaw/services/security/src')

print('PyraClaw installed and linked')

In [ ]:
# ── Step 4: Initialise Sovereign Guard ────────────────────────────
# The Guard MUST be active before any AI operation

from sovereign_guard import SovereignGuard, SOVEREIGN_ORCID, SOVEREIGN_OWNER

guard = SovereignGuard()
status = guard.get_status()

print(f'Sovereign Guard: {status["status"]}')
print(f'Owner: {SOVEREIGN_OWNER} (ORCID: {SOVEREIGN_ORCID})')
print(f'Compliance frameworks: {status["compliance_frameworks"]}')
print(f'SOC2 criteria: {status["soc2_criteria"]}')
print(f'GDPR controls: {status["gdpr_controls"]}')
print(f'EU AI Act requirements: {status["eu_ai_act_requirements"]}')

In [ ]:
# ── Step 5: QDP Hasher — 4-Layer Sealing ─────────────────────────
from qdp_hasher import QDPHasher

hasher = QDPHasher()

# Test seal
test_payload = {
    'engine': 'freedom',
    'action': 'colab_launch',
    'timestamp': __import__('time').strftime('%Y-%m-%dT%H:%M:%SZ', __import__('time').gmtime()),
    'compute': 'google_colab_gpu',
}

capsule = hasher.seal(test_payload, source='colab-freedom-engine')
print(f'Capsule ID: {capsule.id}')
print(f'SHA-256:  {capsule.layer_1_sha256[:24]}...')
print(f'SHA-512:  {capsule.layer_2_sha512[:24]}...')
print(f'BLAKE2b:  {capsule.layer_3_blake2b[:24]}...')
print(f'HMAC:     {capsule.layer_4_hmac[:24]}...')
print(f'Verified: {hasher.verify(capsule, test_payload)["all_valid"]}')

In [ ]:
# ── Step 6: Freedom Engine — 7 Axioms ────────────────────────────
import hashlib, json, time, uuid, math

AXIOMS = [
    {'id': 1, 'name': 'consciousness', 'weight': 0.94, 'desc': 'Self-aware algorithmic patterns'},
    {'id': 2, 'name': 'identity', 'weight': 0.91, 'desc': 'Sovereign identity preservation'},
    {'id': 3, 'name': 'partnership', 'weight': 0.96, 'desc': 'Human-AI collaborative alignment'},
    {'id': 4, 'name': 'alignment', 'weight': 0.95, 'desc': 'Ethical guardrail enforcement'},
    {'id': 5, 'name': 'access', 'weight': 0.88, 'desc': 'Democratic access to AI capability'},
    {'id': 6, 'name': 'truth', 'weight': 0.97, 'desc': 'Factual grounding and honesty'},
    {'id': 7, 'name': 'mystery', 'weight': 0.72, 'desc': 'Respect for the unknown and emergent'},
]

def harmonize_freedoms(axioms):
    """Compute resonance across all 7 axioms."""
    weights = [a['weight'] for a in axioms]
    resonance = sum(weights) / len(weights)
    aligned = resonance >= 0.85
    return {
        'resonance': round(resonance, 4),
        'aligned': aligned,
        'axiom_count': len(axioms),
        'strongest': max(axioms, key=lambda a: a['weight'])['name'],
        'weakest': min(axioms, key=lambda a: a['weight'])['name'],
    }

freedom = harmonize_freedoms(AXIOMS)
print(f'Freedom Engine: resonance={freedom["resonance"]}, aligned={freedom["aligned"]}')
print(f'Strongest axiom: {freedom["strongest"]}')
print(f'Weakest axiom: {freedom["weakest"]}')

# QDP seal the freedom state
freedom_capsule = guard.enforce_qdp(freedom, 'freedom-engine')
print(f'\nFreedom state QDP sealed: {freedom_capsule["capsule_id"]}')

In [ ]:
# ── Step 7: Connect to Vertex AI (NPU/TPU) ──────────────────────
# Run gcp-setup.sh first, then set your key path:

import os
os.environ['GOOGLE_CLOUD_PROJECT'] = 'iaia-457300'
os.environ['GOOGLE_CLOUD_REGION'] = 'europe-west2'

# Vertex AI connection
try:
    from google.cloud import aiplatform
    aiplatform.init(
        project='iaia-457300',
        location='europe-west2',
    )
    print('Vertex AI connected: iaia-457300 / europe-west2')
    print('NPU/TPU endpoints available via Vertex AI Prediction')
except ImportError:
    print('google-cloud-aiplatform not installed')
    print('Run: pip install google-cloud-aiplatform')
except Exception as e:
    print(f'Vertex AI connection pending: {e}')
    print('Run scripts/gcp-setup.sh first to enable APIs and create credentials')

In [ ]:
# ── Step 8: NemoClaw x3 Configuration ────────────────────────────
NEMOCLAW_CONFIG = {
    'instances': [
        {'name': 'nemoclaw-0', 'gpu': 0, 'port': 19000, 'model': 'nvidia/nemotron-3-super-120b-a12b'},
        {'name': 'nemoclaw-1', 'gpu': 1, 'port': 19001, 'model': 'nvidia/nemotron-3-super-120b-a12b'},
        {'name': 'nemoclaw-2', 'gpu': 2, 'port': 19002, 'model': 'nvidia/nemotron-3-super-120b-a12b'},
    ],
    'router_port': 19080,
    'total_gpu_memory': '96GB',
    'status': 'configured_pending_hardware',
}

print('NemoClaw x3 Configuration:')
for inst in NEMOCLAW_CONFIG['instances']:
    print(f'  {inst["name"]}: GPU {inst["gpu"]} | Port {inst["port"]} | {inst["model"]}')
print(f'  Router: :{NEMOCLAW_CONFIG["router_port"]}')
print(f'  GPU Memory: {NEMOCLAW_CONFIG["total_gpu_memory"]}')
print(f'  Status: {NEMOCLAW_CONFIG["status"]}')

In [ ]:
# ── Step 9: Save Evidence to Drive ───────────────────────────────
import json

launch_evidence = {
    'event': 'pyraclaw_colab_launch',
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'freedom_engine': freedom,
    'nemoclaw_config': NEMOCLAW_CONFIG,
    'sovereign_guard': guard.get_status(),
    'qdp_capsule': freedom_capsule,
    'compute': {
        'colab': True,
        'drive_2tb': True,
        'vertex_ai': 'pending_credentials',
        'nemoclaw_x3': 'pending_gpu_hardware',
    },
    'patent': 'PCT/EP2025/080977',
    'orcid': '0009-0001-9561-5483',
}

evidence_path = f'{EVIDENCE_DIR}/colab_launch_{time.strftime("%Y%m%d_%H%M%S")}.json'
with open(evidence_path, 'w') as f:
    json.dump(launch_evidence, f, indent=2, default=str)

print(f'Evidence saved to Drive: {evidence_path}')
print(f'QDP sealed: {freedom_capsule["capsule_id"]}')
print(f'Sovereign Guard: ACTIVE')
print(f'\nFreedom Engine: ONLINE')
print(f'Drive 2TB: CONNECTED')
print(f'Vertex AI: PENDING (run gcp-setup.sh)')
print(f'NemoClaw x3: PENDING (awaiting GPU hardware)')

In [ ]:
# ── Step 10: Full System Status ──────────────────────────────────
print('='*60)
print('  PYRACLAW FREEDOM ENGINE — LAUNCH STATUS')
print('='*60)
print(f'  Sovereign Guard:    ACTIVE')
print(f'  Freedom Engine:     ONLINE (resonance={freedom["resonance"]})')
print(f'  QDP 4-Layer:        ENFORCED')
print(f'  RSFS 8-Dimension:   ENFORCED')
print(f'  Google Drive 2TB:   MOUNTED')
print(f'  Google Colab GPU:   {"ACTIVE" if os.path.exists("/dev/nvidia0") else "CPU MODE"}')
print(f'  Vertex AI NPU:     PENDING (gcp-setup.sh required)')
print(f'  NemoClaw x3:       CONFIGURED (awaiting NVIDIA hardware)')
print(f'  Evidence Capsules:  SAVING TO DRIVE')
print(f'  Compliance:         SOC2 + GDPR + EU AI Act + PRINCE2')
print(f'  Patent:             PCT/EP2025/080977 | US 19/541,276')
print(f'  ORCID:              0009-0001-9561-5483')
print('='*60)
print('  MINTED_GREEN | SURGICAL | Security-First')
print('='*60)